In [142]:
%load_ext autoreload
%autoreload 2
import cv2 as cv
import numpy as np
import boto3
import os
import math
import threading
import requests

from sqlalchemy import create_engine, Column, Integer, String, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from database import SessionLocal, DetectionEvent
from matplotlib import pyplot as plt
from collections import deque
from datetime import datetime
from botocore.client import Config


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Check if it manages to load video

In [ ]:
cap = cv.VideoCapture("test2.mov")

if not cap.isOpened():
    print("Error opening video")

# read the first frame if there is a frame ret = true
ret, frame = cap.read()

### ROI definition

In [ ]:
def roi(frame):
    pts = np.array([
        [355, 153],   # Top left
        [1578, 113],  # Top right
        [1621, 1020], # Bottom right
        [322, 1013]   # Bottom left
    ])

    # Needed for OpenCV to understand coordinates as a list
    pts = pts.reshape((-1, 1, 2))

    # Makes black and white mask:
    mask = np.zeros(frame.shape, dtype = np.uint8)
    cv.fillPoly(mask, [pts], (255,255,255))

    # Black = delete video pixels of original frame; White = let pixels through
    rect_img = cv.bitwise_and(frame, mask)

    return rect_img

### MOG2 definition

In [145]:
def mog2(frame, fgbg_mog2):
    fgmask = fgbg_mog2.apply(frame)

    # Size of the filter
    # TODO: adjust kernel in real world situation
    kernel = np.ones((5,5), np.uint8)

    fgmask_erode = cv.erode(fgmask, kernel, iterations = 1)
    fgmask_final = cv.dilate(fgmask_erode, kernel, iterations = 1)

    return fgmask_final

### Object detection

In [146]:
def object_detect(mask, color_frame, active_trajectories):
    contours, hierarchy = cv.findContours(mask, cv.RETR_TREE, cv.CHAIN_APPROX_NONE)
    
    current_frame_centroids = []
    threat_detected = False  
    
    if len(contours) > 15:
        #print("Camera shake or massive interference detected! Ignoring frame.")
        return False, color_frame, []

    # Find all centroids
    for contour in contours:
        area = cv.contourArea(contour)
        if area > 50:
            x, y, w, h = cv.boundingRect(contour)
            cx = x + (w // 2)
            cy = y + (h // 2)
            current_frame_centroids.append((cx, cy))
            
            cv.rectangle(color_frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Match new centroids to existing trajectories
    new_trajectories = []
    for cx, cy in current_frame_centroids:
        matched = False
        for trail in active_trajectories:
            last_cx, last_cy = trail[-1]
            distance = math.hypot(cx - last_cx, cy - last_cy)
            
            if distance < 75:  # MAX_PIXEL_JUMP
                trail.append((cx, cy))
                if len(trail) > 30:  # MAX_TRAIL_LENGTH
                    trail.pop(0)
                    
                new_trajectories.append(trail)
                active_trajectories.remove(trail)
                matched = True
                break
                
        if not matched:
            new_trajectories.append([(cx, cy)])


    for trail in new_trajectories:
        # Check if this specific trail passes the negative filters
        is_dangerous = is_threat(trail)
        
        if is_dangerous:
            threat_detected = True
            line_color = (0, 0, 255) # Red line for threats
        else:
            line_color = (255, 0, 0) # Blue line for ignored noise (leaves/bugs)

        # Draw the trajectory arc
        for i in range(1, len(trail)):
            cv.line(color_frame, trail[i-1], trail[i], line_color, 2)

    return threat_detected, color_frame, new_trajectories

### Trajectory tracking ("Negative filter")

In [147]:
def is_threat(trail):
    # If not tracked long enough
    if len(trail) < 5:  
        return False
        
    max_speed = 0
    total_distance = 0
    
    for i in range(1, len(trail)):
        x1, y1 = trail[i-1]
        x2, y2 = trail[i]
        
        speed = math.hypot(x2 - x1, y2 - y1) 
        if speed > max_speed:
            max_speed = speed
            
        total_distance += speed

    start_x, start_y = trail[0]
    end_x, end_y = trail[-1]
    displacement = math.hypot(end_x - start_x, end_y - start_y)
    
    if total_distance == 0:
        return False
        
    smoothness = displacement / total_distance
    
    # The Leaf/Wind debris
    # It moves slowly (max_speed < 10) AND flutters around (smoothness < 0.5)
    if max_speed < 10 and smoothness < 0.5:
        return False # Ignore it
        
    # The Hovering Bug
    # It has a long trail (lots of total_distance) but hasn't really gone anywhere (low displacement)
    if total_distance > 200 and displacement < 30:
        return False # Ignore it

    return True

### Generate unique filename with a timestamp

In [148]:
def generate_timestamp():
    timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    return f"event_{timestamp_str}.mp4"

### Initialize and return a new OpenCV VideoWriter

In [149]:
def start_video_writer(filename, fourcc, fps, dimensions):
    return cv.VideoWriter(filename, fourcc, fps, dimensions)

### Handling of processed video

In [150]:
def process_finished_video(filename):

    #Uploads the video to AWS S3, logs the URL to Neon, and deletes the local file.
  
    time_str = filename.replace("event_", "").replace(".mp4", "")

    bucket_name = os.getenv("AWS_BUCKET_NAME")
    
    try:
        print(f"[{filename}] Uploading to AWS S3...")
        
        # Upload to S3 (Local File, Bucket Name, Destination File Name)
        s3_client.upload_file(filename, bucket_name, filename)
        
        # Construct the public S3 URL
        s3_url = s3_client.generate_presigned_url(
            'get_object',
            Params={'Bucket': bucket_name, 'Key': filename},
            ExpiresIn=604800
        )
        print(f"[{filename}] Upload successful! Secure URL generated.")
        
        # Log the cloud URL to your Neon Database
        session = SessionLocal()
        try:
            new_event = DetectionEvent(
                detected_at=time_str, 
                video_filepath=s3_url  # Save the cloud URL
            )
            session.add(new_event)
            session.commit()
            print(f"[{filename}] Logged to Neon PostgreSQL.")
        except Exception as db_err:
            session.rollback()
            print(f"Database error: {db_err}")
        finally:
            session.close()

        discord_url = os.getenv("DISCORD_WEBHOOK_URL")
        if discord_url:
            discord_data = {
                "content": f"Security Alert! Threat recorded at {time_str}.\n Watch here: {s3_url}"
            }
            try:
                # Send the message to Discord
                response = requests.post(discord_url, json=discord_data)
                response.raise_for_status()
                print(f"[{filename}] Discord notification sent!")
            except Exception as discord_err:
                print(f"Discord notification failed: {discord_err}")
       
        # Clean up: Delete the local file
        os.remove(filename)
        print(f"[{filename}] Local file deleted to save space.")

    except Exception as e:
        print(f"Failed to process {filename}: {e}")

### Main loop

In [151]:
pre_roll_buffer = deque(maxlen=100)
filter_buffer = deque()

# for MOG2 background subtraction
fgbg_mog2 = cv.createBackgroundSubtractorMOG2(history=500, varThreshold=100, detectShadows=False)
# VideoWriter needs exact width, height and FPS to create a valid video
width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS)

#fourcc - four character code: used to identify which codec program should use to shirnk the file
fourcc = cv.VideoWriter_fourcc(*'mp4v')


##---------TESTING THE MASK---------##
#out_filter = cv.VideoWriter("out_filter.mp4", fourcc, fps, (width, height), isColor=False)

print("Starting video feed...")

object_detected = False
is_recording = False
out = None
frame_count = 0
frames_since_last_detection = 0
current_filename = "" 
dimensions = (width, height)
active_trajectories = [] # Store lists of (cx, cy) points for every moving object
MAX_TRAIL_LENGTH = 30    # How many frames of history to draw
MAX_PIXEL_JUMP = 75      # Max distance an object can move between frames

s3_client = boto3.client(
    's3',
    region_name='eu-north-1',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    config=Config(signature_version='s3v4') 
)


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Checking that MOG2 stabilized, stops counting after so the counter doesn't run to infinity
    if frame_count < 50:
        frame_count += 1 
    
    rect_image = roi(frame)
        

    # MOG2 logic
    final_mask = mog2(rect_image, fgbg_mog2)

    ##---------TESTING THE MASK---------##
    #out_filter.write(final_mask)
    
    # Run the detection
    is_threat_present, tracked_frame, active_trajectories = object_detect(final_mask, frame, active_trajectories)


    # Recording 60 frames after detecting
    if is_threat_present:
        #In case of two objects being thrown closely after one another, make two videos
        if is_recording and frames_since_last_detection > 15:
            print(f"Rapid second throw! Cutting from {current_filename}...")
            out.release()

            threading.Thread(target=process_finished_video, args=(current_filename,)).start()

            current_filename = generate_timestamp()
            out = start_video_writer(current_filename, fourcc, fps, dimensions)
            
        frames_since_last_detection = 0 

    elif is_recording:
        frames_since_last_detection += 1 

   # If not recording frames go to the buffer, if it is recording they go to hard drive
    if not is_recording:
        pre_roll_buffer.append(tracked_frame)
        
    else:
        out.write(tracked_frame)

    # Trigger the "alarm", object has been detectet 
    # Frame count > 30 to give MOG2 time to stabalize
    if is_threat_present and frame_count > 30 and not is_recording: 
        is_recording = True 
        current_filename = generate_timestamp()

        print(f"Threat detected! Triggering recording to {current_filename}")

        out = start_video_writer(current_filename, fourcc, fps, dimensions)
       
        for buffered_frame in pre_roll_buffer:
            out.write(buffered_frame)
            
        pre_roll_buffer.clear()

    # Stoping the alarm 
    if is_recording and frames_since_last_detection > 60:
        is_recording = False
        # out.release so it can be sent as a video
        out.release() 
        out = None 
        frames_since_last_detection = 0

        #TODO: notification feature here
        # Use threads so that camera doesn't freeze
        threading.Thread(target=process_finished_video, args=(current_filename,)).start()

# Only for PC similuation, real camera feed won't get here on normal run
if out is not None:
    out.release()
    print("Color video saved as 'out.mp4'.")

##---------TESTING THE MASK---------##
#out_filter.release()

cap.release() 
print("Finished completely. Exiting...")

Starting video feed...
Threat detected! Triggering recording to event_2026-11-15_21-50-47.mp4
[event_2026-11-15_21-50-47.mp4] Uploading to AWS S3...
[event_2026-11-15_21-50-47.mp4] Upload successful! Secure URL generated.
[event_2026-11-15_21-50-47.mp4] Logged to Neon PostgreSQL.
[event_2026-11-15_21-50-47.mp4] Discord notification sent!
[event_2026-11-15_21-50-47.mp4] Local file deleted to save space.
Threat detected! Triggering recording to event_2026-11-15_21-51-07.mp4
[event_2026-11-15_21-51-07.mp4] Uploading to AWS S3...
[event_2026-11-15_21-51-07.mp4] Upload successful! Secure URL generated.
[event_2026-11-15_21-51-07.mp4] Logged to Neon PostgreSQL.
[event_2026-11-15_21-51-07.mp4] Discord notification sent!
[event_2026-11-15_21-51-07.mp4] Local file deleted to save space.
Finished completely. Exiting...
